# Team shape and physical load from football tracking

**Question.** How do team shape and physical output differ in and out of possession? Can shape features predict possession phase in a match the model has never seen?

Work through the sections in order, or use **Runtime → Run all** on a standard CPU runtime. Section 1 can download the six Metrica CSVs from GitHub or read them from a ZIP. Figures and tables appear as you run the notebook. The last section exports the results.

The wearable-data review motivated the held-out match test. The dissertation motivated the interest in ball movement. The ball’s x and y coordinates support the pitch check. Modelling ball flight and the Magnus effect would require 3D tracking with height.

## 1. Get the data

Leave **DATA_SOURCE** set to **GitHub** to download the original CSVs from the pinned repository revision. To use a ZIP, select **ZIP** and run the cell to pick your archive. You can also upload it through Colab’s Files sidebar first and set **ZIP_PATH** to its path.

The ZIP needs the three CSVs from each of these folders. You can preserve the folders or put all six files at the root. The loader matches the original filenames.

| Folder | Files |
| --- | --- |
| `data/Sample_Game_1/` | `Sample_Game_1_RawEventsData.csv`, `Sample_Game_1_RawTrackingData_Home_Team.csv`, `Sample_Game_1_RawTrackingData_Away_Team.csv` |
| `data/Sample_Game_2/` | `Sample_Game_2_RawEventsData.csv`, `Sample_Game_2_RawTrackingData_Home_Team.csv`, `Sample_Game_2_RawTrackingData_Away_Team.csv` |

Data credit goes to **Metrica Sports**. The pinned repository has no standalone or named licence. Its README asks for responsible use and source attribution for public work. [Source repository and usage terms](https://github.com/metrica-sports/sample-data/blob/e706dd506b360d69d9d123d5b8026e7294b13996/README.md#legal-stuff) · [Event definitions](https://github.com/metrica-sports/sample-data/blob/e706dd506b360d69d9d123d5b8026e7294b13996/documentation/events-definitions.pdf)

In [ ]:
import csv
from pathlib import Path
from tempfile import NamedTemporaryFile
from urllib.request import urlopen

import numpy as np
import pandas as pd
from scipy.signal import savgol_filter
from scipy.spatial import ConvexHull, QhullError
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import shutil
from zipfile import ZipFile

from IPython.display import Image, display

import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Arc, Circle, Polygon, Rectangle
from matplotlib.ticker import MaxNLocator, PercentFormatter


SOURCE_COMMIT = "e706dd506b360d69d9d123d5b8026e7294b13996"
SOURCE = f"https://raw.githubusercontent.com/metrica-sports/sample-data/{SOURCE_COMMIT}/data"
TEAMS = ("Home", "Away")
FPS = 25
PITCH = np.array([105.0, 68.0])
WINDOW = 15  # Odd number of samples for a centred Savitzky–Golay derivative
MAX_SPEED = 12.0
HSR_SPEED = 5.5
SPRINT_SPEED = 7.0
BRIDGE_SECONDS = 10
FEATURES = ["centroid_x_m", "centroid_y_m", "length_m", "width_m",
            "hull_area_m2", "defensive_line_m"]

INK = "#152b31"
MUTED = "#476067"
PAPER = "#f9faf7"
GRID = "#dbe4de"
TEAM_COLOURS = {"Home": "#156b72", "Away": "#a84a23"}
MODELS = {"Majority baseline": "#74858a", "Logistic regression": "#156b72", "Random forest": "#a84a23"}

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 10.5,
    "figure.facecolor": PAPER, "axes.facecolor": PAPER,
    "text.color": INK, "axes.labelcolor": INK, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.spines.top": False, "axes.spines.right": False, "axes.spines.left": False,
    "axes.edgecolor": GRID, "axes.titleweight": "bold", "savefig.facecolor": PAPER,
})
GAMES = (1, 2)
SUFFIXES = (
    "RawEventsData",
    "RawTrackingData_Home_Team",
    "RawTrackingData_Away_Team",
)
OUTPUT = Path("tracking_results")
FIGURES = OUTPUT / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

In [ ]:
def game_file(data_dir, game, suffix, download):
    name = f"Sample_Game_{game}"
    path = data_dir / name / f"{name}_{suffix}.csv"
    if path.exists() or not download:
        return path
    path.parent.mkdir(parents=True, exist_ok=True)
    print(f"Downloading {path.name}", flush=True)
    with NamedTemporaryFile(dir=path.parent, delete=False) as temp:
        temp_path = Path(temp.name)
        try:
            with urlopen(f"{SOURCE}/{name}/{path.name}", timeout=90) as response:
                while chunk := response.read(1024 * 1024):
                    temp.write(chunk)
            temp.close()
            temp_path.replace(path)
        finally:
            temp_path.unlink(missing_ok=True)
    return path



def extract_match_csvs(archive_path, data_dir):
    # Match original filenames and extract only the required files.
    with ZipFile(archive_path) as archive:
        members = {}
        for member in archive.infolist():
            if not member.is_dir():
                members.setdefault(Path(member.filename).name, []).append(member)
        paths = [game_file(data_dir, game, suffix, download=False)
                 for game in GAMES for suffix in SUFFIXES]
        missing = [path.name for path in paths if path.name not in members]
        if missing:
            raise ValueError("Missing from ZIP: " + ", ".join(missing))
        ambiguous = [path.name for path in paths if len(members[path.name]) != 1]
        if ambiguous:
            raise ValueError("ZIP contains duplicate copies of: " + ", ".join(ambiguous))
        for path in paths:
            path.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(members[path.name][0]) as source, path.open("wb") as dest:
                shutil.copyfileobj(source, dest)

In [ ]:
# @title Choose where to get the match CSVs
DATA_SOURCE = "GitHub"  # @param ["GitHub", "ZIP"]
ZIP_PATH = "metrica_sample_games_1_2_data.zip"  # @param {type:"string"}

if DATA_SOURCE == "GitHub":
    data_dir = Path("metrica_data") / "github"
    for game in GAMES:
        for suffix in SUFFIXES:
            game_file(data_dir, game, suffix, download=True)
elif DATA_SOURCE == "ZIP":
    data_dir = Path("metrica_data") / "zip"
    archive_path = Path(ZIP_PATH)
    if not archive_path.is_file():
        from google.colab import files

        uploaded = files.upload()
        archive_names = [name for name in uploaded if name.lower().endswith(".zip")]
        if len(archive_names) != 1:
            raise ValueError("Select one ZIP containing the six match CSVs.")
        archive_path = Path(archive_names[0])
        del uploaded
    extract_match_csvs(archive_path, data_dir)
else:
    raise ValueError("Choose GitHub or ZIP as DATA_SOURCE.")

print(f"Data ready in {data_dir}")

## 2. Read tracking and check the pitch

The first two tracking rows are metadata. Read paired x/y columns, convert the 0–1 coordinates to a centred **105 × 68 m** pitch and flip y so it points up. Missing coordinates stay missing. Match Home and Away on the frame clock and check that their ball coordinates agree.

Use the first 10 seconds of each half to infer which player is the goalkeeper and which goal the team defends. The snapshot keeps the original pitch orientation so you can check both directions.

In [ ]:
def read_tracking(path):
    with path.open(newline="") as file:
        rows = csv.reader(file)
        next(rows)
        next(rows)
        header = next(rows)
    players = [column.replace(" ", "") for column in header[3:-2:2]]
    columns = ["period", "frame", "time_s"]
    coordinates = [f"{name}_{axis}" for name in [*players, "Ball"] for axis in ("x", "y")]
    raw = pd.read_csv(path, skiprows=3, names=columns + coordinates)
    clock = raw[columns]
    points = raw[coordinates].to_numpy().reshape(len(raw), -1, 2)
    points = (points - 0.5) * PITCH * [1, -1]
    points[~np.isfinite(points).all(axis=2)] = np.nan
    return clock, players, points[:, :-1], points[:, -1]


def infer_team_setup(clock, points):
    setup = {}
    for period, half in clock.groupby("period", sort=False):
        xy = points[half.index]
        calibration = xy[:FPS * 10, :, 0]
        present = np.isfinite(calibration).all(axis=0)
        x = calibration[:, present]
        offsets = np.median(x - np.median(x, axis=1)[:, None], axis=0)
        deepest_player = np.abs(offsets).argmax()
        keeper = np.flatnonzero(present)[deepest_player]
        direction = 1 if offsets[deepest_player] < 0 else -1
        setup[period] = (keeper, direction)
    return setup



def load_game(game, data_dir):
    tracking = {
        team: read_tracking(game_file(data_dir, game, f"RawTrackingData_{team}_Team", download=False))
        for team in TEAMS
    }
    clock, _, _, ball = tracking["Home"]
    away_clock, _, _, away_ball = tracking["Away"]
    if not clock.equals(away_clock) or not np.allclose(ball, away_ball, equal_nan=True):
        raise ValueError(f"Game {game}: Home and Away tracking are out of sync.")
    events = pd.read_csv(game_file(data_dir, game, "RawEventsData", download=False))
    setup = {team: infer_team_setup(clock, tracking[team][2]) for team in TEAMS}
    for period in setup["Home"]:
        if setup["Home"][period][1] == setup["Away"][period][1]:
            raise ValueError(f"Game {game}, half {period}: check attack directions.")
    return clock, tracking, events.fillna({"Subtype": ""}), setup

In [ ]:
matches = {game: load_game(game, data_dir) for game in GAMES}

for game, (clock, tracking, events, setup) in matches.items():
    print(f"Game {game}: {len(clock):,} tracking frames and {len(events):,} events")
    for team in TEAMS:
        roles = [f"{tracking[team][1][keeper]} ({'right' if direction > 0 else 'left'})"
                 for keeper, direction in setup[team].values()]
        print(f"  {team} goalkeeper and attack direction, by half: {', '.join(roles)}")

### Label possession from the events

Pass and shot intervals provide team evidence. Recoveries, restarts, and the start of a loss or ball-out action give additional anchors. Bridge gaps up to 10 seconds between consecutive same-team anchors when the first action supports continuity and no stoppage or uncertain recovery intervenes. Leave conflicting frames and unsupported gaps unlabelled.

Check how much of each match the rule covers. That coverage matters when interpreting the figures and scores.

In [ ]:
def possession_labels(clock, events):
    """Return 1 for Home, 2 for Away and 0 where possession is unlabelled."""
    evidence = np.zeros(len(clock), dtype=np.uint8)
    for period, tracking in clock.groupby("period", sort=False):
        frame = tracking["frame"].to_numpy()
        offset = tracking.index[0]
        period_events = events[events["Period"].eq(period)]
        anchors = []
        stops = []
        for team, kind, subtype, start, end in period_events[
            ["Team", "Type", "Subtype", "Start Frame", "End Frame"]
        ].itertuples(index=False, name=None):
            tags = set(subtype.split("-"))
            end = max(start, end)  # Game 1 kick-off has an end frame of zero
            if kind == "FAULT RECEIVED":
                stops.append(start)
                continue
            if kind == "BALL LOST" and tags & {"OFFSIDE", "FORCED", "END HALF"}:
                stops.append(end)
                continue
            if kind == "BALL OUT" or (kind == "SHOT" and tags & {"GOAL", "OUT"}):
                stops.append(end)
            if kind not in {"PASS", "SHOT", "RECOVERY", "SET PIECE", "BALL LOST", "BALL OUT"}:
                continue
            if kind == "RECOVERY" and tags & {"BLOCKED", "SAVED"}:
                stops.append(start)  # A touch or save can leave a loose ball
                continue
            bit = 1 << TEAMS.index(team)
            last = max(start, end - 1) if kind in {"PASS", "SHOT"} else start
            can_continue = kind not in {"BALL LOST", "BALL OUT"}
            anchors.append((start, last, bit, can_continue))

        stops = np.array(sorted(set(stops)), dtype=int)
        anchors.sort()
        for start, last, bit, _ in anchors:
            left, right = np.searchsorted(frame, [start, last + 1]) + offset
            evidence[left:right] |= bit
        for previous, current in zip(anchors, anchors[1:]):
            _, previous_end, previous_team, can_continue = previous
            start, _, team, _ = current
            if not can_continue or previous_team != team:
                continue
            if not 1 < start - previous_end <= FPS * BRIDGE_SECONDS:
                continue
            stop = np.searchsorted(stops, previous_end)
            if stop < len(stops) and stops[stop] <= start:
                continue
            left, right = np.searchsorted(frame, [previous_end + 1, start]) + offset
            evidence[left:right] |= team
        stop_rows = np.isin(frame, stops)
        evidence[tracking.index[stop_rows]] = 0
    evidence[evidence == 3] = 0  # Conflicting team evidence
    return evidence



def labels_for_team(possession, team):
    return np.select(
        [possession == 0, possession == 1 + TEAMS.index(team)],
        ["Unlabelled", "In possession"], default="Out of possession",
    )


possession_by_game = {
    game: possession_labels(clock, events)
    for game, (clock, _, events, _) in matches.items()
}
for game, possession in possession_by_game.items():
    print(f"Game {game}: {np.count_nonzero(possession):,} labelled frames "
          f"({np.mean(possession != 0):.1%} coverage)")

In [ ]:
def heading(fig, title, subtitle):
    fig.text(0.10, 0.945, title, fontsize=22, weight="bold")
    fig.text(0.10, 0.896, subtitle, fontsize=11, color=MUTED)


def save(fig, destination):
    fig.savefig(destination, dpi=180)
    plt.close(fig)


def draw_pitch(ax):
    line = {"fill": False, "edgecolor": "#80958b", "linewidth": 0.9, "zorder": 1}
    ax.add_patch(Rectangle((-52.5, -34), 105, 68, **line))
    ax.axvline(0, ymin=2 / 72, ymax=70 / 72, color=line["edgecolor"], lw=0.9)
    ax.add_patch(Circle((0, 0), 9.15, **line))
    ax.scatter([0, -41.5, 41.5], [0, 0, 0], s=5, color=line["edgecolor"], zorder=1)
    for side in (-1, 1):
        for depth, width in ((16.5, 40.32), (5.5, 18.32)):
            start = -52.5 if side == -1 else 52.5 - depth
            ax.add_patch(Rectangle((start, -width / 2), depth, width, **line))
        arc = (307, 53) if side == -1 else (127, 233)
        ax.add_patch(Arc((side * 41.5, 0), 18.3, 18.3, theta1=arc[0], theta2=arc[1], **line))
    ax.set(xlim=(-55, 55), ylim=(-36, 36), aspect="equal", xticks=[], yticks=[])
    ax.spines["bottom"].set_visible(False)


def plot_snapshot(clock, tracking, setup, ball, possession, out):
    # Choose the labelled frame nearest 10 minutes with the ball visible.
    options = np.flatnonzero((possession != 0) & np.isfinite(ball).all(axis=1))
    row = options[np.abs(clock.loc[options, "time_s"].to_numpy() - 600).argmin()]
    period, frame, time = clock.loc[row]
    side = list(TEAM_COLOURS)[possession[row] - 1]
    minutes, seconds = divmod(int(time), 60)
    fig, ax = plt.subplots(figsize=(10.2, 7.5))
    fig.subplots_adjust(left=0.10, right=0.90, top=0.86, bottom=0.12)
    heading(fig, "Team shape at a glance",
            f"Sample Game 1  ·  {minutes:02d}:{seconds:02d}  ·  Frame {int(frame):,}  ·  {side} in possession")
    draw_pitch(ax)
    handles = []
    for team, (_, _, points, _) in tracking.items():
        keeper, direction = setup[team][period]
        color = TEAM_COLOURS[team]
        xy = points[row]
        field = np.delete(xy, keeper, axis=0)
        field = field[np.isfinite(field).all(axis=1)]
        hull = field[ConvexHull(field).vertices]
        ax.add_patch(Polygon(hull, color=color, alpha=0.15, zorder=2))
        ax.plot(*np.vstack([hull, hull[0]]).T, color=color, alpha=0.75, lw=1.2, zorder=3)
        ax.scatter(*field.T, s=34, color=color, edgecolor=PAPER, linewidth=0.6, zorder=4)
        ax.scatter(*xy[keeper], s=36, marker="s", facecolor=PAPER, edgecolor=color, zorder=4)
        ax.scatter(*field.mean(axis=0), s=90, marker="+", color=color, linewidth=2, zorder=5)
        handles.append(Line2D([], [], color=color, marker="o", linestyle="none",
                              label=f"{team} attacks {'right' if direction > 0 else 'left'}"))
    ax.scatter(*ball[row], s=45, color=INK, edgecolor="white", linewidth=1.1, zorder=6)
    handles.extend([
        Line2D([], [], color=MUTED, marker="s", markerfacecolor=PAPER, linestyle="none", label="Goalkeeper"),
        Line2D([], [], color=MUTED, marker="+", linestyle="none", label="Centroid"),
        Line2D([], [], color=INK, marker="o", linestyle="none", label="Ball"),
    ])
    fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.50, 0.065),
               ncol=5, frameon=False, columnspacing=1.1, handletextpad=0.35)
    fig.text(0.10, 0.036, "Pitch 105 × 68 m. Shading shows the outfield convex hull.", color=MUTED, fontsize=9.5)
    save(fig, out / "01_team_shape.png")



clock, tracking, _, setup = matches[1]
plot_snapshot(clock, tracking, setup, tracking["Home"][3], possession_by_game[1], FIGURES)
display(Image(filename=str(FIGURES / "01_team_shape.png"), width=850))

## 3. Measure physical load

Estimate both velocity components with a centred 15-frame quadratic Savitzky–Golay derivative. Speed is the magnitude of velocity. Exclude estimates over 12 m/s and windows spanning a raw step over 12 m/s. The filter stops at missing coordinates, tracking discontinuities and half-time.

Distance sums speed × 0.04 s. High-speed running (HSR) uses speeds above 5.5 m/s, and sprint distance uses speeds above 7 m/s. Sprint distance is included in HSR. Count a sprint bout when speed stays above 7 m/s for at least one continuous second, assigning the bout to the five-minute block where it began. Block clocks restart in each half. Use observed player-minutes to compare the two phases.

In [ ]:
def runs(mask):
    edges = np.flatnonzero(np.diff(mask, prepend=False, append=False))
    return zip(edges[::2], edges[1::2])


def speed_for_team(points, clock):
    """Estimate speed only inside continuous, finite, same-period segments."""
    speeds = np.full(points.shape[:2], np.nan)
    rejected = 0
    radius = WINDOW // 2
    for _, half in clock.groupby("period", sort=False):
        first, last = half.index[[0, -1]]
        xy = points[first:last + 1]
        frames = half["frame"].to_numpy()
        elapsed = half["time_s"].to_numpy()
        continuous = (np.diff(frames) == 1) & np.isclose(np.diff(elapsed), 1 / FPS)
        cuts = np.r_[0, np.flatnonzero(~continuous) + 1, len(half)]
        for start, end in zip(cuts, cuts[1:]):
            for player, positions in enumerate(xy[start:end].transpose(1, 0, 2)):
                for left, right in runs(np.isfinite(positions).all(axis=1)):
                    if right - left < WINDOW:
                        continue
                    segment = positions[left:right]
                    velocity = savgol_filter(segment, WINDOW, 2, deriv=1, delta=1 / FPS, axis=0)
                    centres = np.arange(radius, len(segment) - radius)
                    smoothed = np.linalg.norm(velocity[centres], axis=1)
                    jumps = np.linalg.norm(np.diff(segment, axis=0), axis=1) * FPS > MAX_SPEED
                    affected = np.convolve(jumps, np.ones(WINDOW - 1, dtype=int), mode="valid") > 0
                    artefact = (smoothed > MAX_SPEED) | affected
                    rejected += np.count_nonzero(artefact)
                    smoothed[artefact] = np.nan
                    speeds[first + start + left + centres, player] = smoothed
    return speeds, int(rejected)


def distances(speed):
    return {
        "distance_m": speed.sum() / FPS,
        "hsr_m": speed[speed > HSR_SPEED].sum() / FPS,
        "sprint_m": speed[speed > SPRINT_SPEED].sum() / FPS,
    }


def measure_load(clock, game, team, players, speeds, setup, team_labels):
    blocks = []
    phases = []
    for period, half in clock.groupby("period", sort=False):
        idx = half.index
        speed = speeds[idx]
        keeper, _ = setup[period]
        seconds = half["time_s"].to_numpy() - half["time_s"].iloc[0]
        # Round to frame ticks before dividing so floating-point clock errors
        # cannot move a frame to the previous 5-minute block.
        block = np.rint(seconds * FPS).astype(int) // (300 * FPS)
        for i, player in enumerate(players):
            v = speed[:, i]
            valid = np.isfinite(v)
            if not valid.any():
                continue
            sprint_starts = np.array(
                [start for start, end in runs(v > SPRINT_SPEED) if end - start >= FPS],
                dtype=int,
            )
            for bin_id in np.unique(block[valid]):
                sample = v[valid & (block == bin_id)]
                blocks.append({
                    "game": game, "period": period, "block": bin_id + 1,
                    "start_min_in_half": 5 * bin_id, "team": team, "player": player,
                    "role": "Goalkeeper" if i == keeper else "Outfield",
                    "observed_s": len(sample) / FPS, **distances(sample),
                    "sprint_bouts": int(np.count_nonzero(block[sprint_starts] == bin_id)),
                })
        phase = team_labels[idx]
        outfield = np.delete(speed, keeper, axis=1)
        for label in ("In possession", "Out of possession"):
            selected = outfield[phase == label]
            sample = selected[np.isfinite(selected)]
            phases.append({
                "game": game, "team": team, "phase": label,
                "observed_s": len(sample) / FPS, **distances(sample),
            })
    return pd.DataFrame(blocks), pd.DataFrame(phases)

In [ ]:
block_tables = []
phase_tables = []
speed_quality = {}
for game, (clock, tracking, _, setup) in matches.items():
    for team, (_, players, points, _) in tracking.items():
        speeds, rejected = speed_for_team(points, clock)
        speed_quality[game, team] = (int(np.isfinite(speeds).sum()), rejected)
        team_labels = labels_for_team(possession_by_game[game], team)
        blocks, phases = measure_load(clock, game, team, players, speeds, setup[team], team_labels)
        block_tables.append(blocks)
        phase_tables.append(phases)

player_load = pd.concat(block_tables, ignore_index=True)
load_by_phase = pd.concat(phase_tables, ignore_index=True).groupby(
    ["game", "team", "phase"], as_index=False,
).sum(numeric_only=True)
for metric in ("distance", "hsr", "sprint"):
    load_by_phase[f"{metric}_m_per_player_min"] = (
        60 * load_by_phase[f"{metric}_m"] / load_by_phase["observed_s"]
    )

display(player_load.head(8).round(2))
display(load_by_phase[["game", "team", "phase", "distance_m_per_player_min",
                       "hsr_m_per_player_min", "sprint_m_per_player_min"]].round(2))

In [ ]:
def plot_load(load, out):
    sample = load[load["game"].eq(1) & load["role"].eq("Outfield")]
    blocks = sample.groupby(["period", "start_min_in_half", "team"], as_index=False)[
        ["observed_s", "distance_m", "hsr_m", "sprint_m"]
    ].sum()
    fig, axes = plt.subplots(3, 2, figsize=(10.4, 7.9), sharex=True, sharey="row")
    fig.subplots_adjust(left=0.11, right=0.95, top=0.81, bottom=0.13, hspace=0.35, wspace=0.13)
    heading(fig, "Physical load across five-minute blocks",
            "Sample Game 1  ·  Outfield distance per observed player-minute")
    for (column, name), row in zip(
        [("distance_m", "Total distance"), ("hsr_m", "HSR > 5.5 m/s"), ("sprint_m", "Sprint > 7 m/s")],
        axes,
    ):
        for period, ax in enumerate(row, start=1):
            for team, color in TEAM_COLOURS.items():
                data = blocks[blocks["period"].eq(period) & blocks["team"].eq(team)]
                ax.plot(data["start_min_in_half"], data[column] * 60 / data["observed_s"],
                        color=color, lw=1.7, marker="o", ms=4)
            ax.set_ylim(bottom=0)
            ax.yaxis.set_major_locator(MaxNLocator(4))
            ax.set_axisbelow(True)
            ax.grid(axis="y", color=GRID)
            ax.tick_params(axis="both", length=0)
            if period == 1:
                ax.set_ylabel(f"{name}\n(m/player-min)", fontsize=9.5, labelpad=12)
    for period, ax in enumerate(axes[0], start=1):
        ax.set_title("First half" if period == 1 else "Second half", loc="left", pad=10)
    axes[-1, 0].set_xticks([0, 10, 20, 30, 40, 45])
    fig.supxlabel("Block start, minutes into half", x=0.53, y=0.075, fontsize=10.5)
    fig.legend([Line2D([], [], color=c, lw=2, marker="o") for c in TEAM_COLOURS.values()], list(TEAM_COLOURS),
               ncol=2, loc="upper right", bbox_to_anchor=(0.956, 0.878), frameon=False)
    fig.text(0.10, 0.031, "Includes all tracked phases and partial final blocks. Sprint distance is included in HSR.",
             color=MUTED, fontsize=9.5)
    save(fig, out / "02_load_by_block.png")



plot_load(player_load, FIGURES)
display(Image(filename=str(FIGURES / "02_load_by_block.png"), width=850))

## 4. Compute 25 Hz team shape

Rotate each team to attack toward positive x. For every frame with ten valid outfielders, compute centroid x and y, length and width as axis ranges, and convex hull area. Defensive line height is the median longitudinal position of the deepest four outfielders, measured in metres from their own goal line.

The table compares possession phases within each side and match. Positive values indicate larger measurements in possession.

In [ ]:
def describe_shape(clock, game, team, players, points, team_labels, setup):
    """Orient each team's outfield positions towards positive x."""
    tables = []
    for period, half in clock.groupby("period", sort=False):
        xy = points[half.index]
        keeper, direction = setup[period]

        outfield = np.delete(xy, keeper, axis=1) * direction
        valid = np.isfinite(outfield).all(axis=2)
        n_players = valid.sum(axis=1)
        values = np.full((len(half), len(FEATURES)), np.nan)
        complete = n_players == 10
        # Boolean indexing is safe because every selected frame has ten rows.
        coords = outfield[complete][valid[complete]].reshape(-1, 10, 2)
        values[complete, :2] = coords.mean(axis=1)
        values[complete, 2:4] = np.ptp(coords, axis=1)
        deepest = np.partition(coords[:, :, 0], 3, axis=1)[:, :4]
        values[complete, 5] = np.median(deepest, axis=1) + PITCH[0] / 2
        for row, frame in zip(np.flatnonzero(complete), coords):
            try:
                values[row, 4] = ConvexHull(frame).volume  # 2D hull area
            except QhullError:
                values[row, 4] = np.nan

        table = half.assign(
            game=game, team=team, phase=team_labels[half.index], outfield_players=n_players,
            goalkeeper=players[keeper], attack_direction=direction,
        )
        tables.append(table.join(pd.DataFrame(values, index=half.index, columns=FEATURES)))
    return pd.concat(tables)

In [ ]:
shape_by_game = {}
quality = []
for game, (clock, tracking, _, setup) in matches.items():
    print(f"Computing 25 Hz shape for Game {game}", flush=True)
    tables = []
    possession = possession_by_game[game]
    for team, (_, players, points, _) in tracking.items():
        team_labels = labels_for_team(possession, team)
        shape = describe_shape(clock, game, team, players, points, team_labels, setup[team])
        tables.append(shape)
        speed_samples, rejected = speed_quality[game, team]
        quality.append({
            "game": game, "team": team, "frames": len(clock),
            "labelled_frames": np.count_nonzero(possession),
            "in_possession_frames": np.count_nonzero(team_labels == "In possession"),
            "complete_shape_frames": shape[FEATURES].notna().all(axis=1).sum(),
            "speed_samples": speed_samples, "speed_samples_rejected": rejected,
        })
    shape_by_game[game] = pd.concat(tables, ignore_index=True)

team_shape = pd.concat(shape_by_game.values(), ignore_index=True)
shape_by_phase = team_shape[team_shape["phase"].ne("Unlabelled")].groupby(
    ["game", "team", "phase"], as_index=False,
)[FEATURES].mean()
by_phase = shape_by_phase.set_index(["game", "team", "phase"])[FEATURES]
differences = by_phase.xs("In possession", level="phase") - by_phase.xs("Out of possession", level="phase")
display(differences.round(2))

## 5. Test possession prediction across matches

Use only the six shape columns as predictors, including the centroid and line height that capture pitch position. Keep both teams from each eligible instant together and take one frame per second for fitting and scoring. Each instant contributes one positive and one negative row, so the constant majority-class baseline scores 50% accuracy.

Avoid random frame splits because neighbouring tracking frames are near-duplicates. Within Game 1, divide the match clock into five contiguous windows. Use the four later windows for validation in order, training on earlier observations with a one-minute gap. Then fit on all eligible Game 1 rows and evaluate on Game 2. The split verifies transfer to a second match. Anonymisation leaves any overlap in team identity unknown.

In [ ]:
def classifier_rows(shape):
    usable = shape[shape["phase"].ne("Unlabelled")].dropna(subset=FEATURES)
    paired = usable.groupby("frame")["team"].transform("size") == 2
    rows = usable[paired].sort_values(["frame", "team"]).copy()
    rows["target"] = rows["phase"].eq("In possession").astype(int)
    return rows


def score_models(train, test, evaluation, fold):
    models = {
        "Majority baseline": DummyClassifier(strategy="most_frequent"),
        "Logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=500)),
        "Random forest": RandomForestClassifier(
            n_estimators=160, max_depth=9, min_samples_leaf=30, n_jobs=2, random_state=42,
        ),
    }
    x_train, x_test = train[FEATURES], test[FEATURES]
    results = []
    for name, model in models.items():
        model.fit(x_train, train["target"])
        probabilities = model.predict_proba(x_test)[:, 1]
        results.append({
            "evaluation": evaluation, "fold": fold, "model": name,
            "train_game": train["game"].iloc[0], "test_game": test["game"].iloc[0],
            "train_start_frame": train["frame"].min(), "train_end_frame": train["frame"].max(),
            "test_start_frame": test["frame"].min(), "test_end_frame": test["frame"].max(),
            "train_rows": len(train), "test_rows": len(test),
            "accuracy": accuracy_score(test["target"], model.predict(x_test)),
            "roc_auc": roc_auc_score(test["target"], probabilities),
        })
    return results


def evaluate(game1, game2):
    train_match, test_match = map(classifier_rows, (game1, game2))
    # Use one frame per second for fitting and scoring. Split on the original
    # clock before filtering labels so the test windows are contiguous in time.
    train_match = train_match[train_match["frame"].mod(FPS).eq(0)]
    test_match = test_match[test_match["frame"].mod(FPS).eq(0)]
    edges = np.linspace(0, game1["time_s"].max(), 6)
    results = []
    for fold, (start, end) in enumerate(zip(edges[1:-1], edges[2:]), start=1):
        train = train_match[train_match["time_s"] < start - 60]
        test = train_match[train_match["time_s"].between(start, end, inclusive="right")]
        print(f"Game 1 blocked validation, fold {fold}/4", flush=True)
        results.extend(score_models(train, test, "Game 1 blocked CV", fold))
    print("Train Game 1, test Game 2", flush=True)
    results.extend(score_models(train_match, test_match, "Game 1 to Game 2", 1))
    return pd.DataFrame(results)

In [ ]:
scores = evaluate(shape_by_game[1], shape_by_game[2])
summary = scores.groupby(["evaluation", "model"])[["accuracy", "roc_auc"]].mean()
display(summary.assign(
    accuracy=summary["accuracy"].map("{:.1%}".format),
    roc_auc=summary["roc_auc"].map("{:.3f}".format),
))

In [ ]:
def plot_results(scores, out):
    cross = scores[scores["evaluation"].eq("Game 1 to Game 2")].set_index("model")
    fig, axes = plt.subplots(1, 2, figsize=(10.4, 4.8), sharey=True)
    fig.subplots_adjust(left=0.24, right=0.94, top=0.72, bottom=0.20, wspace=0.25)
    heading(fig, "Possession prediction on an unseen match",
            "Models trained on Sample Game 1 and evaluated on Sample Game 2")
    models = list(MODELS)
    for ax, (metric, title) in zip(axes, [("accuracy", "Accuracy"), ("roc_auc", "ROC AUC")]):
        values = cross.loc[models, metric]
        ys = range(len(models))
        ax.barh(ys, values, color=list(MODELS.values()), height=0.46, zorder=3)
        for y, value in zip(ys, values):
            ax.text(value + 0.024, y, f"{value:.1%}", va="center", fontsize=10.5, weight="bold")
        ax.axvline(0.5, lw=1, ls="--", color=INK, alpha=0.65, zorder=4)
        ax.set(xlim=(0, 1), xticks=[0, 0.25, 0.5, 0.75, 1], yticks=list(ys), yticklabels=models)
        ax.set_title(title, loc="left", pad=12)
        ax.xaxis.set_major_formatter(PercentFormatter(1, decimals=0))
        ax.tick_params(axis="both", length=0)
        ax.set_axisbelow(True)
        ax.grid(axis="x", color=GRID)
    axes[0].invert_yaxis()
    fig.text(0.10, 0.071, f"{int(cross['test_rows'].iloc[0]):,} team-frame rows from one frame per second. Dashed line marks 50%.",
             fontsize=9.5, color=MUTED)
    save(fig, out / "03_cross_game_results.png")



plot_results(scores, FIGURES)
display(Image(filename=str(FIGURES / "03_cross_game_results.png"), width=850))

### Interpret the results

The reference run found teams were **10.5–13.7 m wider in possession**. High-speed running per player-minute was higher out of possession in three of four team-match comparisons. On held-out Game 2, the reference logistic regression reached **82.5%** accuracy and the random forest **83.2%**, against the **50.0%** majority baseline. Small differences may occur across software versions.

Use your output to check which side reverses the high-speed-running pattern and whether sprint distance follows the same direction. Label coverage was **44.6%** in Game 1 and **48.2%** in Game 2. The event-based rule favours clearer on-ball sequences, and the resulting rows remain temporally correlated. With two anonymised matches and 2D ball tracking, the scope is descriptive and limited to a single held-out match.

## 6. Export the results

Run the cell to package the three figures, model scores, coverage, 25 Hz shape features and load summaries. Set **DOWNLOAD_RESULTS** to true to download the results ZIP when this cell runs. You can also download it from the Files sidebar.

In [ ]:
# @title Save the analysis outputs
DOWNLOAD_RESULTS = False  # @param {type:"boolean"}

tables = {
    "player_load_5min": player_load,
    "load_by_phase": load_by_phase,
    "shape_by_phase": shape_by_phase,
    "model_scores": scores,
    "data_quality": pd.DataFrame(quality),
}
for name, table in tables.items():
    table.to_csv(OUTPUT / f"{name}.csv", index=False, float_format="%.6f")
team_shape.to_csv(OUTPUT / "team_shape_frames.csv.gz", index=False, float_format="%.3f")

results_zip = shutil.make_archive("team_shape_tracking_results", "zip", root_dir=OUTPUT)
print(f"Saved {Path(results_zip).name}")

if DOWNLOAD_RESULTS:
    from google.colab import files

    files.download(results_zip)